In [1]:
# S4: Window Functions
# Window functions perform calculations across a set of rows that are related to the current row
# GROUP BY collapses rows
# WINDOW functions allow rows to keep their identity while also aggregating information about their grouping

# Without window function (GROUP BY):     With window function:
# supplier   total_spend                  po_id  supplier  amount  supplier_total
# Acme       819898                       PO-001 Acme      32055   819898
# GlobalCo   752095                       PO-002 Acme      22382   819898
# FastParts  995412                       PO-003 GlobalCo  47315   752095
#                                         PO-004 FastParts 23470   995412


# WINDOW FUNCTION documentation

# function_name() OVER (
#     PARTITION BY column    -- define the groups (like GROUP BY)
#     ORDER BY column        -- define the order within each group
#     ROWS/RANGE BETWEEN ... -- optional: define the window frame
# )

# PARTITION BY — divides rows into groups. Like GROUP BY but rows aren't collapsed

# ORDER BY — defines the order within each partition — required for ranking and running totals

# OVER() — what makes it a window function. Without OVER() it's a regular aggregate



In [2]:
# Six Important Window Functions to Practice

# ROW_NUMBER() — unique sequential number per partition -- no ties allowed -- assigns a unique # to each row

# RANK() — ranking with gaps for ties -- tied values get the same rank -- next rank is skipped [2,2,2,4] <-- no 3

# DENSE_RANK() — ranking without gaps for ties -- applied consecutive tiers no skips [2,2,2,3,4,4,5]

# LAG() — access a previous row's value -- returns the value from N rows AFTER the current row

# LEAD() — access a next row's value -- returns the value from N rows AFTER the current row

# SUM() OVER() — running total -- cumulative sum within a partition based on a columns order

# High Level Summary

# GROUP BY collapses rows -- SC example would be collapsing all rows so there is 1 per Supplier or SKU or Warehouse
# PARTITION BY keeps all rows -- SC example would add a total alongside each row/instance in your data

In [3]:
import pandas as pd
import numpy as np
%load_ext sql
%sql duckdb:///:memory:

np.random.seed(22)
suppliers = ["Acme", "GlobalCo", "FastParts", "PrimeMg", "EastCoast"]
categories = ["Electronics", "Hardware", "Consumables", "Apparel"]
warehouses = ["East", "West", "Central"]

rows = []
for i in range(1, 201):
    rows.append({
        "po_id":          f"PO-{str(i).zfill(4)}",
        "supplier":       np.random.choice(suppliers),
        "category":       np.random.choice(categories),
        "warehouse":      np.random.choice(warehouses),
        "order_date":     pd.Timestamp("2024-01-01") + pd.Timedelta(days=int(np.random.randint(0, 365))),
        "amount":         round(np.random.uniform(500, 50000), 2),
        "units":          np.random.randint(10, 500),
        "lead_time_days": np.random.randint(3, 45),
        "on_time":        np.random.choice([True, False], p=[0.8, 0.2])
    })

po = pd.DataFrame(rows)
%sql --persist po
print(f"po: {po.shape}")

Connecting to 'duckdb:///:memory:'

Running query in 'duckdb:///:memory:'

Success! Persisted po to the database.

po: (200, 9)


In [4]:
po.head(5)

,po_id,supplier,category,warehouse,order_date,amount,units,lead_time_days,on_time
0,PO-0001,EastCoast,Electronics,East,2024-12-22,32055.22,94,11,True
1,PO-0002,FastParts,Consumables,East,2024-02-15,23470.66,103,42,True
2,PO-0003,Acme,Electronics,West,2024-01-28,22382.64,39,21,True
3,PO-0004,GlobalCo,Apparel,Central,2024-01-08,47315.60,33,26,True
4,PO-0005,EastCoast,Apparel,East,2024-12-25,37347.63,123,4,False


In [5]:
%%sql

-- S4.1 ROW_NUMBER()
-- Business question: what is the highest value PO per supplier?
-- step 1: use ROW_NUMBER() to rank POs within each supplier by amount descending
-- step 2: wrap in a subquery and filter where row_num = 1
-- show: supplier, po_id, amount, row_num

select
    supplier,
    po_id,
    amount,
    row_num
from (
select
    supplier,
    po_id,
    amount,
    ROW_NUMBER() OVER(
        PARTITION BY supplier -- restarts the ranking per unique supplier
        ORDER BY amount DESC
    ) AS row_num
from
    po
)
where
    row_num = 1 -- keeps only the top row per supplier
order by
    amount desc -- ensures highest ranking is the first row


Running query in 'duckdb:///:memory:'

supplier,po_id,amount,row_num
EastCoast,PO-0155,49202.14,1
Acme,PO-0083,49123.34,1
PrimeMg,PO-0046,47855.66,1
GlobalCo,PO-0045,47711.66,1
FastParts,PO-0050,47326.18,1


In [6]:
%%sql

-- S4.2 RANK() and DENSE_RANK()

-- ?: How do suppliers rank by total spend?
-- Show both RANK() and DENSE_RANK() side-by-side

-- OVER is the keyword in the WINDOW FUNCTION
-- When using RANK() SQL needs to know what to RANK() OVER()
-- OVER() opens a window -- which is a defined set of rows
-- without OVER the aggregate functions SUM, AVG, RANK would collapse all rows into one

SELECT
    supplier,
    total_spend,
    RANK() OVER (ORDER BY total_spend DESC) AS spend_rank,
    DENSE_RANK() OVER (ORDER BY total_spend DESC) AS spend_dense_rank
FROM (
    SELECT
        supplier,
        ROUND(SUM(amount), 2) AS total_spend
    FROM
        po
    GROUP BY
        supplier
)
ORDER BY spend_rank



Running query in 'duckdb:///:memory:'

supplier,total_spend,spend_rank,spend_dense_rank
EastCoast,1200154.42,1,1
PrimeMg,1071116.69,2,2
FastParts,995411.84,3,3
Acme,819898.26,4,4
GlobalCo,752094.86,5,5


In [7]:
%%sql

-- S4.3 SUM() OVER() running total
-- Business question: what is the cumulative spend per supplier over time?

SELECT
    supplier,
    order_date,
    amount,
    ROUND(SUM(amount) OVER(
        PARTITION BY supplier -- divides the dataset into groups -- resetting and recalculating per group
        ORDER BY order_date -- defines the order within the partition
    ), 2) AS running_total
FROM
    po
ORDER BY
    order_date, running_total
LIMIT 20


Running query in 'duckdb:///:memory:'

supplier,order_date,amount,running_total
FastParts,2024-01-05 00:00:00,6239.84,6239.84
FastParts,2024-01-06 00:00:00,8790.67,15030.51
GlobalCo,2024-01-08 00:00:00,47315.6,47315.6
GlobalCo,2024-01-09 00:00:00,16459.99,63775.59
FastParts,2024-01-13 00:00:00,41305.05,56335.56
EastCoast,2024-01-14 00:00:00,4627.14,4627.14
EastCoast,2024-01-17 00:00:00,12485.09,17112.23
FastParts,2024-01-17 00:00:00,19609.79,75945.35
Acme,2024-01-18 00:00:00,39190.33,39190.33
EastCoast,2024-01-18 00:00:00,24930.62,42042.85


In [8]:
%%sql

-- S4.4 LAG() -- to generate Month-Over-Month Change
-- Business question: is each suppliers spend trending up or down month over month?

SELECT
    supplier,
    month, -- generate this field from order_date
    total_spend, --
    prev_spend, --
    ROUND(total_spend - prev_spend, 2) AS change,
    CASE
        WHEN total_spend > prev_spend THEN 'Up'
        WHEN total_spend < prev_spend THEN 'Down'
        ELSE 'Flat'
    END AS trend
FROM (
    SELECT
        supplier,
        DATE_TRUNC('month', order_date) AS month,
        ROUND(SUM(amount), 2) AS total_spend,
        LAG(ROUND(SUM(amount), 2)) OVER (
            PARTITION BY 
                supplier
            ORDER BY
                DATE_TRUNC('month', order_date)
        ) AS prev_spend
    FROM
        po
    GROUP BY
        supplier,
        DATE_TRUNC('month', order_date)
)
ORDER BY supplier, month
LIMIT 20

Running query in 'duckdb:///:memory:'

supplier,month,total_spend,prev_spend,change,trend
Acme,2024-01-01 00:00:00,121571.9,None,None,Flat
Acme,2024-03-01 00:00:00,115261.79,121571.9,-6310.11,Down
Acme,2024-04-01 00:00:00,35049.25,115261.79,-80212.54,Down
Acme,2024-05-01 00:00:00,66368.49,35049.25,31319.24,Up
Acme,2024-07-01 00:00:00,67316.87,66368.49,948.38,Up
Acme,2024-08-01 00:00:00,157175.55,67316.87,89858.68,Up
Acme,2024-09-01 00:00:00,5104.86,157175.55,-152070.69,Down
Acme,2024-10-01 00:00:00,15369.59,5104.86,10264.73,Up
Acme,2024-11-01 00:00:00,112842.26,15369.59,97472.67,Up
Acme,2024-12-01 00:00:00,123837.7,112842.26,10995.44,Up


In [9]:
po.head(5)

,po_id,supplier,category,warehouse,order_date,amount,units,lead_time_days,on_time
0,PO-0001,EastCoast,Electronics,East,2024-12-22,32055.22,94,11,True
1,PO-0002,FastParts,Consumables,East,2024-02-15,23470.66,103,42,True
2,PO-0003,Acme,Electronics,West,2024-01-28,22382.64,39,21,True
3,PO-0004,GlobalCo,Apparel,Central,2024-01-08,47315.60,33,26,True
4,PO-0005,EastCoast,Apparel,East,2024-12-25,37347.63,123,4,False


In [12]:
%%sql

-- S4.5 AVG() OVER() -- PO vs. Supplier Average
-- Business question: which POs are significantly above or below their supplier's average?

SELECT
    po_id,
    supplier,
    amount,
    ROUND(AVG(amount) OVER (PARTITION BY supplier), 2) AS supplier_avg, -- average amount by supplier
    ROUND(                                                              -- percent difference
        (amount - AVG(amount) OVER (PARTITION BY supplier))
        /
        AVG(amount) OVER (PARTITION BY supplier) * 100
        , 1) AS pct_diff,
CASE                                                                    -- categorize
    WHEN pct_diff > 20 THEN 'high'
    WHEN pct_diff < -20 THEN 'low'
    ELSE 'normal'
END AS size_vs_avg
FROM
    po
ORDER BY
    pct_diff DESC
LIMIT 20

Running query in 'duckdb:///:memory:'

po_id,supplier,amount,supplier_avg,pct_diff,size_vs_avg
PO-0155,EastCoast,49202.14,22225.08,121.4,high
PO-0061,EastCoast,49140.95,22225.08,121.1,high
PO-0167,EastCoast,45104.8,22225.08,102.9,high
PO-0067,EastCoast,44930.02,22225.08,102.2,high
PO-0050,FastParts,47326.18,23700.28,99.7,high
PO-0129,EastCoast,44333.82,22225.08,99.5,high
PO-0196,FastParts,47054.5,23700.28,98.5,high
PO-0046,PrimeMg,47855.66,24909.69,92.1,high
PO-0083,Acme,49123.34,25621.82,91.7,high
PO-0039,PrimeMg,47663.03,24909.69,91.3,high
